# 🎯 CloudSync SaaS Analytics: 03. Customer Segmentation & Health Scoring

**Project:** CloudSync Customer Churn & Retention Analytics  
**Dataset:** CloudSync Enriched Master Profile  
**Objective:** Construct RFM (Recency, Frequency, Monetary) quintile scoring, engineer a multi-factor SaaS Customer Health Score (0–100), define strategic business value segments, and perform K-Means unsupervised behavioral clustering.

---

### Customer Health Score Design Architecture
The CloudSync Health Score evaluates account vitality across 5 weighted business pillars:
1. **Product Engagement (35%)**: Monthly login frequency, active days per month, and breadth of feature utilization.
2. **Satisfaction & Advocacy (25%)**: Net Promoter Score (NPS) and CSAT survey ratings.
3. **Payment Reliability (15%)**: Payment gateway success rate and penalty on payment failures/delinquency.
4. **Support Ticket Friction (15%)**: Resolution velocity and penalty on unresolved/escalated tickets.
5. **Account Tenure (10%)**: Length of subscription tenure stability.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 300

CLEAN_DIR = '../data/cleaned'
df_master = pd.read_csv(os.path.join(CLEAN_DIR, "customer_churn_master.csv"))

print(f"Master feature dataset loaded with {len(df_master)} records.")

## 1. RFM Score Distribution & Quantile Scoring

In [ ]:
# RFM Summary
print("RFM Scores Summary:")
display(df_master[['recency_months', 'avg_monthly_logins', 'total_paid', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].describe())

# Distribution of R, F, M Scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x='R_Score', data=df_master, ax=axes[0], palette='Blues')
axes[0].set_title('Recency Score Distribution (5 = Most Recent)', fontweight='bold')
sns.countplot(x='F_Score', data=df_master, ax=axes[1], palette='Greens')
axes[1].set_title('Frequency Score Distribution (5 = Highest Logins)', fontweight='bold')
sns.countplot(x='M_Score', data=df_master, ax=axes[2], palette='Purples')
axes[2].set_title('Monetary Score Distribution (5 = Highest Spend)', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Customer Health Score Validation & Segment Breakdown

In [ ]:
print("Customer Health Score Summary Statistics:")
display(df_master['health_score'].describe())

print("
Strategic Customer Health Segments:")
display(df_master['customer_health_segment'].value_counts())

# Health Score across Segments
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_master, x='customer_health_segment', y='health_score', palette='tab10', ax=ax)
ax.set_title('Customer Health Score Distribution by Strategic Segment', fontsize=13, fontweight='bold')
ax.set_ylabel('Health Score (0-100)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. High-Value At-Risk Account Identification

In [ ]:
# Filter high-value customers at high risk
high_val_risk = df_master[df_master['customer_health_segment'] == 'High Value – High Risk'].copy()
print(f"Total High Value - High Risk accounts identified: {len(high_val_risk)}")
print(f"Total Monthly MRR at immediate risk: ${high_val_risk['monthly_fee'].sum():,.2f}")
print(f"Annualized MRR at risk: ${high_val_risk['monthly_fee'].sum() * 12:,.2f}")

display(high_val_risk[['customer_id', 'customer_name', 'plan_name', 'monthly_fee', 'health_score', 'avg_monthly_logins', 'failed_payments', 'unresolved_tickets', 'nps_score']].head(10))

## 4. Unsupervised K-Means Behavioral Clustering

In [ ]:
cluster_features = ['avg_monthly_logins', 'avg_session_mins', 'avg_features_used', 'failed_payments', 'nps_score', 'monthly_fee']
X_cluster = df_master[cluster_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_master['cluster_id'] = kmeans.fit_predict(X_scaled)

cluster_profile = df_master.groupby('cluster_id')[cluster_features + ['health_score', 'churn_probability']].mean().round(2)
cluster_profile['Account_Count'] = df_master.groupby('cluster_id')['customer_id'].count()

print("K-Means Behavioral Cluster Profiles:")
display(cluster_profile)